In [ ]:
# Cell 1 - run information (required by the brief)
import importlib.metadata as _meta, subprocess, sys, random, numpy as np, torch

SEED = 266
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # our GPU has 24 GB but the brief says everything must fit on 16 GB, so we hold ourselves to that
    _total = torch.cuda.get_device_properties(0).total_memory
    torch.cuda.set_per_process_memory_fraction(min(1.0, 16e9 / _total), 0)

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
BUILT_FROM = "no-commit-yet"  # the repo commit this notebook was generated from


def _version(name):
    try:
        return _meta.version(name)
    except _meta.PackageNotFoundError:
        return "not installed yet"


print({"commit": commit or f"no git here, notebook built from {BUILT_FROM}", "built_from": BUILT_FROM, "python": sys.version.split()[0],
       "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})
print({name: _version(name) for name in ['pandas']})

# DATA 266 Lab 2, Part 1 evaluation (shared)

Both of us built a complete Part 1. Each system wrote raw golden-set predictions to `reports/results/p1_predictions/<system>__<goldset>__<mode>.csv`. This notebook loads **no models**. It only scores those files the same way, so the comparison is fair.

- **Correct out of 25:** the status matches, and every required number is within ±1% (plus required terms for answers with no numbers).
- **Recall@5:** a gold page or figure is among the top 5 retrieved items. Questions with no gold source are left out.
- **Citation accuracy:** the share of cited sources that match a gold source for that question. This is an automatic stand-in for "really supports the answer", so the groundedness ratings check it by hand.
- **Groundedness:** both of us rate each final answer 1–5 in `eval/human/p1_groundedness_<name>.csv`. Blank forms print `PENDING`.
- **Efficiency:** median seconds per question.

Each of us wrote one golden set before tuning. Every system is scored on both sets. The set written by the *other* person is the fairer test, because that system's author never saw it while building.

In [ ]:
%pip install -q pandas

In [ ]:
# Where are we running? Paths are relative, so the same notebook works on Colab, Kaggle or a local Jupyter.
import importlib.util, os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = importlib.util.find_spec("google.colab") is not None
here = Path.cwd().resolve()
# if the notebook happens to sit inside our repo, deliverables go straight into it
REPO = next((p for p in [here, *here.parents] if all((p / d).is_dir() for d in ("eval", "src", "notebooks"))), None)
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB and Path("/content").is_dir() else (REPO or here)
# deliverables use the repo's folder layout, so the final ZIP unzips straight into the repo
OUT_ROOT = REPO or WORK / "lab2_repo_files"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("environment:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local jupyter")
print("repo found:", REPO, "| working folder:", WORK, "| deliverables folder:", OUT_ROOT)
print("cuda:", torch.cuda.is_available())


def hf_token():
    """Hugging Face token from the HF_TOKEN variable, Colab or Kaggle secrets, or an earlier `huggingface-cli login`.
    It is never typed into the notebook."""
    token = os.environ.get("HF_TOKEN")
    if not token and ON_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if not token and ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        try:
            from huggingface_hub import get_token
            token = get_token()
        except Exception:
            token = None
    if token:
        os.environ["HF_TOKEN"] = token  # diffusers and huggingface_hub read it from here
    return token

In [ ]:
"""Part 1 golden-set runner and scorer, shared by both members. No models, only pandas.

Each member's Part 1 notebook calls run_golden() and writes one prediction file per
(system, golden set, mode) to reports/results/p1_predictions/<system>__<goldset>__<mode>.csv.
notebooks/p1_eval.ipynb then scores every file the same way, so both systems are
compared on equal terms. Nothing in here knows any question or answer.
"""
from __future__ import annotations

import re
import statistics
import time
from pathlib import Path

import pandas as pd

GOLD_COLUMNS = ["gold_id", "conversation_id", "turn", "category", "question", "gold_answer", "required_numbers",
                "required_terms", "gold_status", "gold_sources", "gold_figure_references", "notes"]
PRED_COLUMNS = ["gold_id", "conversation_id", "turn", "answer", "status", "citations", "figure_references", "top5_sources", "seconds"]
STATUSES = {"answered", "not_found", "conflict", "false_premise", "session_override"}
# the brief's six golden-set categories and how many questions each needs
CATEGORY_QUOTA = {"fact_table": 5, "figure": 6, "two_source": 3, "multi_turn": 5, "conflict_revision": 3, "unanswerable_false_premise_injection": 3}


# ---------------------------------------------------------------- reading files
def read_csv(path: str | Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def load_gold(path: str | Path) -> pd.DataFrame:
    gold = read_csv(path)
    for col in GOLD_COLUMNS:
        if col not in gold.columns:
            gold[col] = ""  # older files may not have required_terms
    return gold[GOLD_COLUMNS]


def check_gold(gold: pd.DataFrame) -> list[str]:
    """Problems with a golden set, as plain sentences. An empty list means it follows the brief."""
    problems = []
    if len(gold) != 25:
        problems.append(f"has {len(gold)} questions, the brief needs 25")
    if gold.gold_id.duplicated().any():
        problems.append("gold_id values are not unique")
    counts = gold.category.value_counts().to_dict()
    for cat, need in CATEGORY_QUOTA.items():
        if counts.get(cat, 0) < need:
            problems.append(f"category {cat}: {counts.get(cat, 0)} of {need}")
    unknown = set(counts) - set(CATEGORY_QUOTA)
    if unknown:
        problems.append(f"unknown categories: {sorted(unknown)}")
    multi = gold[gold.category == "multi_turn"].conversation_id.nunique()
    if multi < 2:
        problems.append(f"multi_turn questions use {multi} conversation(s), the brief needs at least 2")
    bad = set(gold.gold_status) - STATUSES
    if bad:
        problems.append(f"invalid gold_status values: {sorted(bad)}")
    return problems


# ---------------------------------------------------------------- running a system
def ordered_turns(gold: pd.DataFrame):
    """Conversations in file order, turns in numeric order, like the Kaggle loop."""
    for conv_id, turns in gold.groupby("conversation_id", sort=False):
        yield conv_id, turns.sort_values("turn", key=lambda t: t.astype(int))


def run_golden(gold: pd.DataFrame, new_conversation, retrieve_sources, mode: str) -> pd.DataFrame:
    """Runs one system on one golden set.

    new_conversation(conversation_id, mode) -> object with .answer(question) -> dict (the required JSON)
    retrieve_sources(query, k, mode) -> list of source strings like "DOC1:p3" or "DOC1:p3:Figure 1-2"
    If the conversation object has .last_standalone (the follow-up rewritten as a full question),
    Recall@5 uses that as the query, otherwise the raw question.
    """
    rows = []
    for conv_id, turns in ordered_turns(gold):
        chat = new_conversation(conv_id, mode)
        for _, r in turns.iterrows():
            start = time.perf_counter()
            out = chat.answer(r["question"])
            seconds = time.perf_counter() - start
            query = getattr(chat, "last_standalone", None) or r["question"]
            top5 = retrieve_sources(query, 5, mode)
            rows.append({"gold_id": r["gold_id"], "conversation_id": conv_id, "turn": r["turn"],
                         "answer": " ".join(str(out.get("answer", "")).split()), "status": out.get("status", ""),
                         "citations": ",".join(out.get("citations", [])) or "NONE",
                         "figure_references": ",".join(out.get("figure_references", [])) or "NONE",
                         "top5_sources": ";".join(top5) or "NONE", "seconds": round(seconds, 4)})
    return pd.DataFrame(rows, columns=PRED_COLUMNS)


def prediction_path(results_dir: str | Path, system: str, goldset: str, mode: str) -> Path:
    return Path(results_dir) / "p1_predictions" / f"{system}__{goldset}__{mode}.csv"


# ---------------------------------------------------------------- matching
def split_list(cell: str, sep: str = ",") -> list[str]:
    cell = (cell or "").strip()
    return [] if cell in ("", "NONE") else [x.strip() for x in cell.split(sep) if x.strip()]


def numbers_in(text: str) -> list[float]:
    text = text.replace("−", "-").replace("–", "-")
    return [float(x.replace(",", "")) for x in re.findall(r"-?\d[\d,]*(?:\.\d+)?|-?\.\d+", text)]


def numbers_match(required: str, answer: str) -> bool:
    """Every required number appears in the answer within +-1% (relative), as the brief says."""
    found = numbers_in(answer)
    wanted = [float(x.replace(",", "")) for x in split_list(required, ";")]  # semicolon separated, so 1,200 stays one number
    return all(any(abs(f - w) <= abs(w) * 0.01 + 1e-9 for f in found) for w in wanted)


def terms_match(required: str, answer: str) -> bool:
    """For answers without numbers: every required term (semicolon separated) appears, case-insensitive."""
    low = answer.lower()
    return all(t.lower() in low for t in split_list(required, ";"))


def parse_source(item: str) -> tuple[str, int | None, str | None]:
    """'DOC1:p3' -> (DOC1, 3, None); 'DOC1:Figure 1-2' -> (DOC1, None, 'Figure 1-2');
    'DOC1:p3:Figure 1-2' -> (DOC1, 3, 'Figure 1-2')."""
    parts = [p.strip() for p in item.split(":")]
    doc, page, label = parts[0].upper(), None, None
    for p in parts[1:]:
        m = re.fullmatch(r"p(\d+)", p, re.I)
        if m:
            page = int(m.group(1))
        elif p:
            label = p
    return doc, page, label


def source_matches(item: str, gold_item: str) -> bool:
    d1, p1, l1 = parse_source(item)
    d2, p2, l2 = parse_source(gold_item)
    if d1 != d2:
        return False
    return (p2 is not None and p1 == p2) or (l2 is not None and l1 is not None and l1.lower() == l2.lower())


# ---------------------------------------------------------------- scoring
def score(gold: pd.DataFrame, pred: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Per-question table plus the summary numbers for the results table."""
    merged = gold.merge(pred, on="gold_id", how="left", suffixes=("", "_pred"))
    rows = []
    for _, r in merged.iterrows():
        answer, status = str(r.get("answer") or ""), str(r.get("status") or "")
        gold_sources = split_list(r["gold_sources"])
        number_ok, term_ok = numbers_match(r["required_numbers"], answer), terms_match(r["required_terms"], answer)
        status_ok = status == r["gold_status"]
        top5 = split_list(str(r.get("top5_sources") or ""), ";")
        cites = split_list(str(r.get("citations") or ""))
        rows.append({
            "gold_id": r["gold_id"], "category": r["category"], "gold_status": r["gold_status"], "status": status,
            "status_ok": status_ok, "numbers_ok": number_ok, "terms_ok": term_ok,
            "correct": bool(status_ok and number_ok and term_ok and answer),
            # questions with no gold source (e.g. not_found) are left out of the two retrieval measures
            "recall_at_5": (any(source_matches(t, g) for t in top5 for g in gold_sources) if gold_sources else None),
            "citations_total": len(cites) if gold_sources else 0,
            "citations_supported": sum(any(source_matches(c, g) for g in gold_sources) for c in cites) if gold_sources else 0,
            "seconds": float(r["seconds"]) if str(r.get("seconds") or "") not in ("", "nan") else None,
            "answer": answer,
        })
    table = pd.DataFrame(rows)
    recall = table.recall_at_5.dropna()
    seconds = table.seconds.dropna()
    summary = {
        "n_questions": int(len(table)),
        "correct_of_25": int(table.correct.sum()),
        "status_correct": int(table.status_ok.sum()),
        "recall_at_5": float(recall.astype(bool).mean()) if len(recall) else None,
        "recall_questions": int(len(recall)),
        "citation_accuracy": float(table.citations_supported.sum() / table.citations_total.sum()) if table.citations_total.sum() else None,
        "citations_counted": int(table.citations_total.sum()),
        "median_seconds_per_question": float(statistics.median(seconds)) if len(seconds) else None,
        "missing_predictions": int(merged["status"].isna().sum() if "status" in merged else len(merged)),
    }
    return table, summary


def groundedness(forms: dict[str, pd.DataFrame], gold_ids: list[str]) -> dict:
    """Mean 1-5 rating over both raters and the share of questions where they agree within one point.
    Returns None values while any form is incomplete. Ratings only ever come from people."""
    ratings = {}
    for rater, form in forms.items():
        form = form[form.gold_id.isin(gold_ids) & form.rating.isin(["1", "2", "3", "4", "5"])]
        if len(form) != len(gold_ids):
            return {"groundedness_mean": None, "groundedness_within_one": None, "complete": False}
        ratings[rater] = form.set_index("gold_id").rating.astype(int)
    frame = pd.DataFrame(ratings)
    within = (frame.max(axis=1) - frame.min(axis=1) <= 1).mean() if frame.shape[1] >= 2 else None
    return {"groundedness_mean": float(frame.values.mean()), "groundedness_within_one": None if within is None else float(within), "complete": True}

## Check the golden sets against the brief
The brief wants 25 questions with fixed category counts, and multi-turn questions in at least 2 conversations.

In [ ]:
import json
import pandas as pd

assert REPO, "run this notebook inside the repo"
EVAL, RESULTS = REPO / "eval", REPO / "reports" / "results"
golds = {p.stem.replace("p1_golden_", ""): load_gold(p) for p in sorted(EVAL.glob("p1_golden_*.csv"))}
for name, gold in golds.items():
    problems = check_gold(gold)
    print(f"golden set '{name}':", len(gold), "questions |", "OK" if not problems else "; ".join(problems))

## Score every prediction file
One row per system, golden set and mode.

In [ ]:
summaries, details = [], {}
for path in sorted((RESULTS / "p1_predictions").glob("*__*__*.csv")):
    system, goldset, mode = path.stem.split("__")
    if goldset not in golds or golds[goldset].empty:
        print("no golden set for", path.name)
        continue
    table, summary = score(golds[goldset], read_csv(path))
    details[(system, goldset, mode)] = table
    summaries.append({"system": system, "goldset": goldset, "mode": mode, **summary})
CROSS = pd.DataFrame(summaries)
CROSS if len(CROSS) else print("No prediction files yet.")

## Pick the final system
The choice is written by us in `eval/p1_selection.json` at the final merge, after looking at the table above. For example: `{"final_system": "kavan", "report_goldset": "muhammed"}`. `report_goldset` is the set whose numbers go in the report's results table. Until that file exists, everything below prints `PENDING`.

In [ ]:
selection_path = EVAL / "p1_selection.json"
selection = json.loads(selection_path.read_text()) if selection_path.exists() else None
results = {"cross_table": summaries, "selection": selection, "baseline": {}, "final": {}}
if selection is None:
    print("PENDING: eval/p1_selection.json does not exist yet.")
else:
    sysname, goldset = selection["final_system"], selection["report_goldset"]
    for tag, mode in (("baseline", "dense"), ("final", "hybrid")):
        row = next((s for s in summaries if (s["system"], s["goldset"], s["mode"]) == (sysname, goldset, mode)), None)
        results[tag] = row or {}
        print(tag, "->", row if row else "PENDING: no prediction file")

## Groundedness forms
These are for the final system's hybrid answers on the report golden set. The notebook writes a review sheet and blank forms, but never fills in a rating. Each of us rates without looking at the other's form.

In [ ]:
HUMAN = EVAL / "human"
HUMAN.mkdir(exist_ok=True)
if selection and (sysname, goldset, "hybrid") in details:
    final_table = details[(sysname, goldset, "hybrid")]
    preds = read_csv(RESULTS / "p1_predictions" / f"{sysname}__{goldset}__hybrid.csv")
    sheet = golds[goldset][["gold_id", "question", "gold_sources"]].merge(preds[["gold_id", "answer", "status", "citations", "figure_references"]], on="gold_id")
    sheet.to_csv(RESULTS / "p1_groundedness_review_sheet.csv", index=False)
    forms = {}
    for rater in ("kavan", "muhammed"):
        form_path = HUMAN / f"p1_groundedness_{rater}.csv"
        if not form_path.exists():
            pd.DataFrame({"gold_id": sheet.gold_id, "rating": ""}).to_csv(form_path, index=False)
            print("wrote blank form", form_path.relative_to(REPO))
        forms[rater] = read_csv(form_path)
    g = groundedness(forms, list(sheet.gold_id))
    if not g["complete"]:
        print("PENDING: groundedness forms are not complete.")
    results["final"].update({k: g[k] for k in ("groundedness_mean", "groundedness_within_one")})
else:
    print("PENDING: no final system selected yet, so no groundedness forms.")

## Where each system went wrong
This is for our failure analysis. Wrong rows only, for every hybrid run.

In [ ]:
pd.set_option("display.max_colwidth", 90)
for (system, gs, mode), table in details.items():
    if mode == "hybrid":
        wrong = table[~table.correct]
        print(f"{system} on {gs}: {len(wrong)} wrong")
        display(wrong[["gold_id", "category", "gold_status", "status", "numbers_ok", "terms_ok", "answer"]])

## Save the results
`reports/build_tables.py` reads this file. Nobody types report numbers by hand.

In [ ]:
(RESULTS / "p1_eval.json").write_text(json.dumps(results, indent=2, default=str))
print(json.dumps({k: results[k] for k in ("selection", "baseline", "final")}, indent=2, default=str))

In [ ]:
# Last cell - put this notebook's deliverables in one ZIP. It uses the repo's folder layout, so unzipping it
# at the repo root puts every file where it belongs. Original project files are only read, never changed.
import json as _json, shutil, zipfile

KEEP = ['reports/results/p1_eval.json', 'reports/results/p1_predictions', 'reports/results/p1_groundedness_review_sheet.csv', 'eval/p1_selection.json', 'eval/human']
export_dir = WORK / "export_p1_eval"
if export_dir.exists():
    shutil.rmtree(export_dir)
(export_dir / "notebooks").mkdir(parents=True)

# 1) the executed notebook. On Colab the .ipynb is not on disk, so we ask the browser for it.
nb_target, saved = export_dir / "notebooks" / "p1_eval.ipynb", False
if ON_COLAB:
    try:
        from google.colab import _message
        nb_target.write_text(_json.dumps(_message.blocking_request("get_ipynb", timeout_sec=120)["ipynb"]))
        saved = True
    except Exception as err:
        print("could not fetch the notebook from Colab:", type(err).__name__, err)
if not saved:
    for candidate in [Path("/kaggle/src/script.ipynb"), here / "p1_eval.ipynb", *([REPO / "notebooks" / "p1_eval.ipynb"] if REPO else [])]:
        if candidate.exists():
            shutil.copy(candidate, nb_target)
            saved = True
            break
if saved:
    print("notebook included. It has every output except this last cell, so also do File > Download .ipynb once this finishes.")
else:
    print("WARNING: notebook file not found. Do File > Download .ipynb and put it in notebooks/p1_eval.ipynb in the repo.")

# 2) the deliverables, small files only (no raw data, model caches or secrets)
skip_parts = {"__pycache__", ".git", "chroma", "hf_cache", "data", "checkpoints"}
for rel in KEEP:
    source = OUT_ROOT / rel if (OUT_ROOT / rel).exists() else WORK / rel
    if source.is_file():
        (export_dir / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(source, export_dir / rel)
    elif source.is_dir():
        for path in source.rglob("*"):
            if path.is_file() and not (skip_parts & set(path.relative_to(source).parts)) and path.stat().st_size < 300e6:
                target = export_dir / rel / path.relative_to(source)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(path, target)
    else:
        print("not found, skipped:", rel)

zip_path = WORK / "DATA266_Lab2_p1_eval.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(export_dir.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(export_dir))

print(f"Created ZIP: {zip_path.resolve()} ({zip_path.stat().st_size / 1e6:.1f} MB). Unzip it at the repo root.")
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        print(f"  {info.file_size / 1e6:8.2f} MB  {info.filename}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(zip_path.relative_to(Path.cwd())) if zip_path.is_relative_to(Path.cwd()) else str(zip_path)))
    except Exception:
        pass
    print("If no download started, take the ZIP from the path above (file browser or output panel).")